<a href="https://colab.research.google.com/github/AngieVenta/AngieVenta/blob/main/DS_C2_SC2_VENTAPEREZ_GLORIAANGELICA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de Datos de Vacunación COVID-19
## Reto de Análisis con Python y Pandas

Este notebook contiene el análisis completo del dataset de vacunación COVID-19.

In [31]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


In [32]:
# Verificar e instalar openpyxl si es necesario
try:
    import openpyxl
    print(f"✓ openpyxl ya está instalado (versión {openpyxl.__version__})")
except ImportError:
    print("⚠️ openpyxl no está instalado. Instalando...")
    !pip install -q openpyxl
    import openpyxl
    print(f"✓ openpyxl instalado correctamente (versión {openpyxl.__version__})")


✓ openpyxl ya está instalado (versión 3.1.5)


## Carga del Archivo CSV

Selecciona una de las dos opciones para cargar el archivo:
- **Opción 1**: Subir archivo manualmente
- **Opción 2**: Cargar desde una ruta específica

In [33]:
# OPCIÓN 1
# Descomenta esta opción si quieres subir el archivo manualmente
"""
from google.colab import files
print("Por favor, selecciona el archivo country_vaccinations.csv")
uploaded = files.upload()
file_path = 'country_vaccinations.csv'
"""

#OPCIÓN 2
#Carga desde sample_data
file_path = '/content/sample_data/country_vaccinations.csv'

## a) Extraer la información del archivo

**Objetivo**: Cargar el archivo CSV y mostrar información básica sobre el dataset.

In [35]:
# Cargar el archivo CSV
df = pd.read_csv(file_path)

print("=" * 60)
print("INFORMACIÓN GENERAL DEL DATASET")
print("=" * 60)
print(f"✓ Archivo cargado exitosamente")
print(f"✓ Total de registros: {len(df):,}")
print(f"✓ Total de columnas: {len(df.columns)}")
print(f"✓ Nombre de las columnas: {list(df.columns)}")

print("\n Primeras 5 filas del dataset:")
print(df.head())

print("\n Últimas 5 filas del dataset:")
print(df.tail())

INFORMACIÓN GENERAL DEL DATASET
✓ Archivo cargado exitosamente
✓ Total de registros: 86,512
✓ Total de columnas: 15
✓ Nombre de las columnas: ['country', 'iso_code', 'date', 'total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated', 'daily_vaccinations_raw', 'daily_vaccinations', 'total_vaccinations_per_hundred', 'people_vaccinated_per_hundred', 'people_fully_vaccinated_per_hundred', 'daily_vaccinations_per_million', 'vaccines', 'source_name', 'source_website']

 Primeras 5 filas del dataset:
       country iso_code        date  total_vaccinations  people_vaccinated  \
0  Afghanistan      AFG  2021-02-22                 0.0                0.0   
1  Afghanistan      AFG  2021-02-23                 NaN                NaN   
2  Afghanistan      AFG  2021-02-24                 NaN                NaN   
3  Afghanistan      AFG  2021-02-25                 NaN                NaN   
4  Afghanistan      AFG  2021-02-26                 NaN                NaN   

   people_fully_vaccin

## b) Mostrar la estructura y tipos de datos

**Objetivo**: Identificar los tipos de datos de cada columna y asegurar que las fechas sean del tipo `datetime64`.

In [36]:
#Estructura de los datos
print("=" * 60)
print("ESTRUCTURA Y TIPOS DE DATOS")
print("=" * 60)

print("\n Información general del DataFrame:")
df.info()

print("\n Tipos de datos actuales:")
print(df.dtypes)

# Convertir la columna 'date' a datetime64
print("\n Convirtiendo columna 'date' a datetime64...")
df['date'] = pd.to_datetime(df['date'])

print("\n✓ Conversión completada")
print(f"✓ Tipo de dato de 'date': {df['date'].dtype}")

print("\n Tipos de datos después de la conversión:")
print(df.dtypes)

print("\n Estadísticas descriptivas de columnas numéricas:")
print(df.describe())

print("\n Valores nulos por columna:")
print(df.isnull().sum())

ESTRUCTURA Y TIPOS DE DATOS

 Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86512 entries, 0 to 86511
Data columns (total 15 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   country                              86512 non-null  object 
 1   iso_code                             86512 non-null  object 
 2   date                                 86512 non-null  object 
 3   total_vaccinations                   43607 non-null  float64
 4   people_vaccinated                    41294 non-null  float64
 5   people_fully_vaccinated              38802 non-null  float64
 6   daily_vaccinations_raw               35362 non-null  float64
 7   daily_vaccinations                   86213 non-null  float64
 8   total_vaccinations_per_hundred       43607 non-null  float64
 9   people_vaccinated_per_hundred        41294 non-null  float64
 10  people_fully_vaccinated_per_h

## c) Cantidad de vacunas aplicadas por compañía

**Objetivo**: Agrupar por la columna `vaccines` y sumar el total de vacunas aplicadas para cada combinación de compañías.

In [37]:
print("=" * 60)
print("VACUNAS APLICADAS POR COMPAÑÍA")
print("=" * 60)

# NOTA IMPORTANTE SOBRE LA METODOLOGÍA:
# El dataset reporta combinaciones de vacunas (ej: "Pfizer/BioNTech, Moderna")
# sin especificar cuántas de cada compañía se aplicaron.
#
# SUPOSICIÓN: Dividimos equitativamente las vacunas entre las compañías
# reportadas en cada registro.
#
# Ejemplo: Si un país reporta "Pfizer/BioNTech, Moderna" con 1000 vacunas,
# asignamos 500 a Pfizer/BioNTech y 500 a Moderna.
#
# Esta es una aproximación razonable dado que no hay datos desagregados
# por compañía en el dataset original.
#
# IMPORTANTE: Usamos solo el último registro de cada país para evitar
# sumar totales acumulados históricos múltiples veces.

print("\n⚠️  METODOLOGÍA: División equitativa de vacunas en combinaciones")
print("-" * 60)
# Obtener el último registro por país (total acumulado más reciente)
last_record_by_country = df.sort_values('date').groupby('country').last().reset_index()

print(f"✓ Usando último registro de cada país: {len(last_record_by_country)} países")

# Crear lista de registros expandidos con división equitativa
expanded_records = []

for idx, row in last_record_by_country.iterrows():
    if pd.notna(row['vaccines']) and pd.notna(row['total_vaccinations']):
        # Separar las vacunas por coma
        vaccines_list = [v.strip() for v in str(row['vaccines']).split(',')]
        num_vaccines = len(vaccines_list)

        # Dividir el total equitativamente entre todas las vacunas
        divided_total = row['total_vaccinations'] / num_vaccines

        # Crear un registro por cada vacuna
        for vaccine in vaccines_list:
            expanded_records.append({
                'vaccine': vaccine,
                'total_vaccinations': divided_total,
                'country': row['country']
            })
# Crear DataFrame con registros expandidos
df_expanded = pd.DataFrame(expanded_records)

# Agrupar por vacuna individual y sumar
vaccines_by_company = df_expanded.groupby('vaccine')['total_vaccinations'].sum().sort_values(ascending=False)

print("\n Total de vacunas por compañía individual:")
print(vaccines_by_company)

print(f"\n✓ Se encontraron {len(vaccines_by_company)} compañías diferentes")
print(f"✓ Total de vacunas distribuidas: {vaccines_by_company.sum():,.0f}")

print(f"\n Top 10 compañías con mayor cantidad de vacunas:")
for i, (vaccine, total) in enumerate(vaccines_by_company.head(10).items(), 1):
    print(f"  {i:2d}. {vaccine:30s} : {total:>20,.0f}")

print(f"\n Bottom 5 compañías con menor cantidad de vacunas:")
for vaccine, total in vaccines_by_company.tail(5).items():
    print(f"      {vaccine:30s} : {total:>20,.0f}")


VACUNAS APLICADAS POR COMPAÑÍA

⚠️  METODOLOGÍA: División equitativa de vacunas en combinaciones
------------------------------------------------------------
✓ Usando último registro de cada país: 223 países

 Total de vacunas por compañía individual:
vaccine
Oxford/AstraZeneca    1.799690e+09
Pfizer/BioNTech       1.335120e+09
Sinovac               1.149288e+09
Sinopharm/Beijing     1.105147e+09
Moderna               9.824346e+08
Sputnik V             9.575298e+08
Johnson&Johnson       8.028995e+08
CanSino               7.581724e+08
Covaxin               6.642804e+08
ZF2001                6.602003e+08
Sinopharm/Wuhan       6.575185e+08
Novavax               1.568707e+08
EpiVacCorona          8.306408e+07
Abdala                5.332148e+07
Turkovac              4.896064e+07
Soberana02            3.577437e+07
Sputnik Light         3.554192e+07
SpikoGen              1.631028e+07
COVIran Barekat       1.631028e+07
FAKHRAVAC             1.631028e+07
Razi Cov Pars         1.631028e+07
Medig

## d) Cantidad de vacunas aplicadas en todo el mundo

**Objetivo**: Calcular el total acumulado de vacunas aplicadas a nivel mundial.

In [38]:
print("=" * 60)
print("VACUNAS APLICADAS EN TODO EL MUNDO")
print("=" * 60)

# Obtener el último registro por país para tener el total acumulado más reciente
last_record_by_country = df.sort_values('date').groupby('country').last()
world_total = last_record_by_country['total_vaccinations'].sum()

print(f"\n🌍 Total de vacunas aplicadas en todo el mundo: {world_total:,.0f}")
print(f"✓ Basado en el último registro de {len(last_record_by_country)} países")


VACUNAS APLICADAS EN TODO EL MUNDO

🌍 Total de vacunas aplicadas en todo el mundo: 11,383,441,962
✓ Basado en el último registro de 223 países


## e) Promedio de vacunas aplicadas por país

**Objetivo**: Calcular el promedio de vacunas aplicadas considerando todos los países.

In [39]:
print("=" * 60)
print("PROMEDIO DE VACUNAS POR PAÍS")
print("=" * 60)

# Obtener el último registro por país para mantener consistencia con apartados C y D
last_record_by_country = df.sort_values('date').groupby('country').last()

# Calcular el promedio de vacunas aplicadas considerando el último registro de cada país
overall_avg = last_record_by_country['total_vaccinations'].mean()

print(f"\n Promedio de vacunas aplicadas por país: {overall_avg:,.2f}")
print(f"✓ Calculado sobre {len(last_record_by_country)} países")
print(f"✓ Basado en el último registro de cada país (total acumulado actual)")

print("\n Top 10 países con mayor cantidad de vacunas:")
top_10 = last_record_by_country['total_vaccinations'].sort_values(ascending=False).head(10)
for i, (country, total) in enumerate(top_10.items(), 1):
    print(f"  {i:2d}. {country:25s} : {total:>20,.0f}")

print("\n Bottom 10 países con menor cantidad de vacunas:")
bottom_10 = last_record_by_country['total_vaccinations'].sort_values(ascending=True).head(10)
for i, (country, total) in enumerate(bottom_10.items(), 1):
    print(f"  {i:2d}. {country:25s} : {total:>20,.0f}")
print("\n Estadísticas adicionales:")
print(f"  • Mediana: {last_record_by_country['total_vaccinations'].median():,.0f}")
print(f"  • Desviación estándar: {last_record_by_country['total_vaccinations'].std():,.0f}")
print(f"  • Mínimo: {last_record_by_country['total_vaccinations'].min():,.0f}")
print(f"  • Máximo: {last_record_by_country['total_vaccinations'].max():,.0f}")


PROMEDIO DE VACUNAS POR PAÍS

 Promedio de vacunas aplicadas por país: 51,046,824.94
✓ Calculado sobre 223 países
✓ Basado en el último registro de cada país (total acumulado actual)

 Top 10 países con mayor cantidad de vacunas:
   1. China                     :        3,263,129,000
   2. India                     :        1,834,500,657
   3. United States             :          560,181,791
   4. Brazil                    :          413,559,595
   5. Indonesia                 :          377,108,938
   6. Japan                     :          254,345,587
   7. Bangladesh                :          243,642,749
   8. Pakistan                  :          219,368,557
   9. Vietnam                   :          203,144,374
  10. Mexico                    :          191,907,868

 Bottom 10 países con menor cantidad de vacunas:
   1. Pitcairn                  :                   94
   2. Tokelau                   :                1,936
   3. Niue                      :                4,161
   4.

## f) Vacunas aplicadas el 29/01/2021

**Objetivo**: Determinar la cantidad total de vacunas aplicadas en todo el mundo en la fecha específica del 29 de enero de 2021.

In [40]:
print("=" * 60)
print("VACUNAS APLICADAS EL 29/01/2021")
print("=" * 60)

specific_date = pd.to_datetime('2021-01-29')
records_jan_29 = df[df['date'] == specific_date]

# Usar daily_vaccinations porque el enunciado pide "aplicadas EL día" (ese día específico)
# no "hasta el día" (acumulado)
total_jan_29_daily = records_jan_29['daily_vaccinations'].sum()
total_jan_29_accumulated = records_jan_29['total_vaccinations'].sum()

print(f"\n Vacunas aplicadas específicamente el 29/01/2021: {total_jan_29_daily:,.0f}")
print(f"✓ Países que reportaron datos ese día: {len(records_jan_29)}")

print(f"\n Para referencia:")
print(f"  • Vacunas del día (daily_vaccinations): {total_jan_29_daily:,.0f}")
print(f"  • Total acumulado hasta esa fecha: {total_jan_29_accumulated:,.0f}")

print(f"\n Top 10 países con más vacunas aplicadas ese día:")
top_countries_daily = records_jan_29.nlargest(10, 'daily_vaccinations')[['country', 'daily_vaccinations']]
for i, (idx, row) in enumerate(top_countries_daily.iterrows(), 1):
    if pd.notna(row['daily_vaccinations']):
        print(f"  {i:2d}. {row['country']:25s} : {row['daily_vaccinations']:>15,.0f}")

print(f"\n Países que reportaron el 29/01/2021:")
countries_list = sorted(records_jan_29['country'].tolist())
print(f"Total: {len(countries_list)} países")
if len(countries_list) <= 20:
    for country in countries_list:
        print(f"  • {country}")

VACUNAS APLICADAS EL 29/01/2021

 Vacunas aplicadas específicamente el 29/01/2021: 4,884,052
✓ Países que reportaron datos ese día: 83

 Para referencia:
  • Vacunas del día (daily_vaccinations): 4,884,052
  • Total acumulado hasta esa fecha: 82,952,931

 Top 10 países con más vacunas aplicadas ese día:
   1. United States             :       1,449,389
   2. China                     :         880,622
   3. United Kingdom            :         361,343
   4. England                   :         310,733
   5. India                     :         301,348
   6. Brazil                    :         208,792
   7. Israel                    :         177,601
   8. Turkey                    :         103,734
   9. United Arab Emirates      :          95,361
  10. Germany                   :          90,617

 Países que reportaron el 29/01/2021:
Total: 83 países


## g) DataFrame conDiferencias

**Objetivo**: Crear un nuevo DataFrame que contenga todos los datos originales más una columna llamada `diferencias` que muestre la diferencia entre `daily_vaccinations` y `daily_vaccinations_raw`.

In [41]:
print("=" * 60)
print("DATAFRAME conDiferencias")
print("=" * 60)

conDiferencias = df.copy()
conDiferencias['diferencias'] = conDiferencias['daily_vaccinations'] - conDiferencias['daily_vaccinations_raw']

print("\n✓ DataFrame 'conDiferencias' creado exitosamente")
print(f"✓ Total de filas: {len(conDiferencias):,}")
print(f"✓ Total de columnas: {len(conDiferencias.columns)}")

print("\n Primeras 10 filas con la nueva columna 'diferencias':")
print(conDiferencias[['country', 'date', 'daily_vaccinations', 'daily_vaccinations_raw', 'diferencias']].head(10))

print(f"\n Estadísticas de la columna 'diferencias':")
print(conDiferencias['diferencias'].describe())

print(f"\n Registros con mayor diferencia positiva:")
print(conDiferencias.nlargest(5, 'diferencias')[['country', 'date', 'diferencias']])


DATAFRAME conDiferencias

✓ DataFrame 'conDiferencias' creado exitosamente
✓ Total de filas: 86,512
✓ Total de columnas: 16

 Primeras 10 filas con la nueva columna 'diferencias':
       country       date  daily_vaccinations  daily_vaccinations_raw  \
0  Afghanistan 2021-02-22                 NaN                     NaN   
1  Afghanistan 2021-02-23              1367.0                     NaN   
2  Afghanistan 2021-02-24              1367.0                     NaN   
3  Afghanistan 2021-02-25              1367.0                     NaN   
4  Afghanistan 2021-02-26              1367.0                     NaN   
5  Afghanistan 2021-02-27              1367.0                     NaN   
6  Afghanistan 2021-02-28              1367.0                     NaN   
7  Afghanistan 2021-03-01              1580.0                     NaN   
8  Afghanistan 2021-03-02              1794.0                     NaN   
9  Afghanistan 2021-03-03              2008.0                     NaN   

   diferencias  

## h) Periodo de tiempo entre registros

**Objetivo**: Calcular el periodo de tiempo entre el registro con la fecha más reciente y el registro con la fecha más antigua.

In [42]:
print("=" * 60)
print("PERIODO DE TIEMPO ENTRE REGISTROS")
print("=" * 60)

most_recent_date = df['date'].max()
oldest_date = df['date'].min()
time_period = most_recent_date - oldest_date

print(f"\n Fecha más antigua: {oldest_date.strftime('%d/%m/%Y')}")
print(f" Fecha más reciente: {most_recent_date.strftime('%d/%m/%Y')}")
print(f"\n  Periodo total: {time_period.days} días")
print(f"   Equivalente a:")
print(f"   - {time_period.days / 7:.1f} semanas")
print(f"   - {time_period.days / 30:.1f} meses (aproximadamente)")
print(f"   - {time_period.days / 365:.2f} años (aproximadamente)")

PERIODO DE TIEMPO ENTRE REGISTROS

 Fecha más antigua: 02/12/2020
 Fecha más reciente: 29/03/2022

  Periodo total: 482 días
   Equivalente a:
   - 68.9 semanas
   - 16.1 meses (aproximadamente)
   - 1.32 años (aproximadamente)


## i) DataFrame conCantidad

**Objetivo**: Crear un nuevo DataFrame con los datos originales y una columna derivada `canVac` que contenga la cantidad de vacunas utilizadas cada día (separando la columna `vaccines` por el carácter `,`).

In [43]:
print("=" * 60)
print("DATAFRAME conCantidad")
print("=" * 60)

conCantidad = df.copy()
# Contar la cantidad de vacunas separadas por coma
conCantidad['canVac'] = conCantidad['vaccines'].apply(lambda x: len(str(x).split(',')) if pd.notna(x) else 0)

print("\n✓ DataFrame 'conCantidad' creado exitosamente")
print(f"✓ Total de filas: {len(conCantidad):,}")
print(f"✓ Columna 'canVac' agregada (cantidad de vacunas por registro)")

print("\n Primeras 10 filas con la nueva columna 'canVac':")
print(conCantidad[['country', 'date', 'vaccines', 'canVac']].head(10))

print(f"\n Distribución de cantidad de vacunas utilizadas:")
print(conCantidad['canVac'].value_counts().sort_index())

print(f"\n Países con mayor variedad de vacunas en un solo registro:")
print(conCantidad.nlargest(5, 'canVac')[['country', 'date', 'vaccines', 'canVac']])


DATAFRAME conCantidad

✓ DataFrame 'conCantidad' creado exitosamente
✓ Total de filas: 86,512
✓ Columna 'canVac' agregada (cantidad de vacunas por registro)

 Primeras 10 filas con la nueva columna 'canVac':
       country       date                                           vaccines  \
0  Afghanistan 2021-02-22  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
1  Afghanistan 2021-02-23  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
2  Afghanistan 2021-02-24  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
3  Afghanistan 2021-02-25  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
4  Afghanistan 2021-02-26  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
5  Afghanistan 2021-02-27  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
6  Afghanistan 2021-02-28  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
7  Afghanistan 2021-03-01  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
8  Afghanistan 2021-03-02  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/B

## j) DataFrame antes20

**Objetivo**: Crear un nuevo DataFrame denominado `antes20` que contenga todos los registros realizados antes del 20 de diciembre de 2020.

In [44]:
print("=" * 60)
print("DATAFRAME antes20")
print("=" * 60)

cutoff_date = pd.to_datetime('2020-12-20')
antes20 = df[df['date'] < cutoff_date].copy()

print(f"\n✓ DataFrame 'antes20' creado exitosamente")
print(f"✓ Total de registros antes del 20/12/2020: {len(antes20):,}")
print(f"✓ Países involucrados: {antes20['country'].nunique()}")
print(f"✓ Rango de fechas: {antes20['date'].min().strftime('%d/%m/%Y')} - {antes20['date'].max().strftime('%d/%m/%Y')}")

print("\n Primeras 10 filas:")
print(antes20.head(10))

print("\n Países con registros antes del 20/12/2020:")
print(antes20['country'].unique())

DATAFRAME antes20

✓ DataFrame 'antes20' creado exitosamente
✓ Total de registros antes del 20/12/2020: 70
✓ Países involucrados: 8
✓ Rango de fechas: 02/12/2020 - 19/12/2020

 Primeras 10 filas:
      country iso_code       date  total_vaccinations  people_vaccinated  \
13403  Canada      CAN 2020-12-14                 5.0                5.0   
13404  Canada      CAN 2020-12-15               727.0              727.0   
13405  Canada      CAN 2020-12-16              3025.0             3025.0   
13406  Canada      CAN 2020-12-17              7279.0             7279.0   
13407  Canada      CAN 2020-12-18             11296.0            11296.0   
13408  Canada      CAN 2020-12-19             12367.0            12367.0   
15756   China      CHN 2020-12-15           1500000.0                NaN   
15757   China      CHN 2020-12-16                 NaN                NaN   
15758   China      CHN 2020-12-17                 NaN                NaN   
15759   China      CHN 2020-12-18           

## k) DataFrame pfizer

**Objetivo**: Crear un DataFrame denominado `pfizer` que contenga todos los registros donde se haya utilizado la vacuna Pfizer.

In [45]:
print("=" * 60)
print("DATAFRAME pfizer")
print("=" * 60)

# Filtrar registros donde se utilizó Pfizer
pfizer = df[df['vaccines'].str.contains('Pfizer', case=False, na=False)].copy()

print(f"\n✓ DataFrame 'pfizer' creado exitosamente")
print(f"✓ Total de registros con Pfizer: {len(pfizer):,}")
print(f"✓ Países que usaron Pfizer: {pfizer['country'].nunique()}")
print(f"✓ Rango de fechas: {pfizer['date'].min().strftime('%d/%m/%Y')} - {pfizer['date'].max().strftime('%d/%m/%Y')}")

print("\n Primeras 10 filas:")
print(pfizer[['country', 'date', 'vaccines', 'total_vaccinations']].head(10))

print("\n Países que utilizaron Pfizer:")
pfizer_countries = pfizer['country'].unique()
print(f"Total: {len(pfizer_countries)} países")
print(sorted(pfizer_countries))

print("\n Total de vacunas por país (basado en último registro):")
pfizer_by_country = pfizer.sort_values('date').groupby('country').last()
top_10_pfizer = pfizer_by_country.nlargest(10, 'total_vaccinations')[['total_vaccinations']]
for i, (country, row) in enumerate(top_10_pfizer.iterrows(), 1):
    print(f"  {i:2d}. {country:25s} : {row['total_vaccinations']:>20,.0f}")
# Cálculo adicional: Estimado de vacunas Pfizer con división equitativa
print("\n" + "-" * 60)
print(" ESTIMADO DE VACUNAS PFIZER (División Equitativa)")
print("-" * 60)
print("⚠️  Aplicando la misma metodología del apartado C:")
print("    Dividimos equitativamente cuando hay combinaciones de vacunas\n")

# Obtener países que usaron Pfizer
countries_with_pfizer = pfizer['country'].unique()

# Obtener último registro de cada país que usó Pfizer
last_records_pfizer = df[df['country'].isin(countries_with_pfizer)].sort_values('date').groupby('country').last()

# Aplicar división equitativa
pfizer_total_estimated = 0
for idx, row in last_records_pfizer.iterrows():
    if pd.notna(row['vaccines']) and pd.notna(row['total_vaccinations']):
        vaccines_list = [v.strip() for v in str(row['vaccines']).split(',')]

        # Verificar si Pfizer está en la lista
        if any('Pfizer' in vaccine for vaccine in vaccines_list):
            num_vaccines = len(vaccines_list)
            divided_total = row['total_vaccinations'] / num_vaccines
            pfizer_total_estimated += divided_total

print(f"✓ Total estimado de vacunas Pfizer/BioNTech: {pfizer_total_estimated:,.0f}")
print(f"✓ Basado en último registro de {len(last_records_pfizer)} países")
print(f"✓ Metodología: División equitativa en combinaciones")

DATAFRAME pfizer

✓ DataFrame 'pfizer' creado exitosamente
✓ Total de registros con Pfizer: 64,193
✓ Países que usaron Pfizer: 158
✓ Rango de fechas: 02/12/2020 - 29/03/2022

 Primeras 10 filas:
       country       date                                           vaccines  \
0  Afghanistan 2021-02-22  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
1  Afghanistan 2021-02-23  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
2  Afghanistan 2021-02-24  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
3  Afghanistan 2021-02-25  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
4  Afghanistan 2021-02-26  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
5  Afghanistan 2021-02-27  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
6  Afghanistan 2021-02-28  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
7  Afghanistan 2021-03-01  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
8  Afghanistan 2021-03-02  Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...   
9  Af

## l) Guardar DataFrames en Excel

**Objetivo**: Almacenar los cuatro DataFrames generados (`conDiferencias`, `conCantidad`, `antes20` y `pfizer`) en un archivo de Excel denominado `resultadosReto.xlsx`, donde cada DataFrame ocupe una hoja diferente.

In [46]:
print("=" * 60)
print("GUARDANDO RESULTADOS EN EXCEL")
print("=" * 60)

# Crear archivo Excel con múltiples hojas usando pd.ExcelWriter
output_filename = 'resultadosReto.xlsx'

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    conDiferencias.to_excel(writer, sheet_name='conDiferencias', index=False)
    conCantidad.to_excel(writer, sheet_name='conCantidad', index=False)
    antes20.to_excel(writer, sheet_name='antes20', index=False)
    pfizer.to_excel(writer, sheet_name='pfizer', index=False)

print(f"\n✓ Archivo '{output_filename}' creado exitosamente")
print("\n Resumen de hojas creadas:")
print(f"   1. conDiferencias - {len(conDiferencias):,} filas")
print(f"   2. conCantidad    - {len(conCantidad):,} filas")
print(f"   3. antes20        - {len(antes20):,} filas")
print(f"   4. pfizer         - {len(pfizer):,} filas")

# Verificar que el archivo se creó correctamente
if os.path.exists(output_filename):
    file_size = os.path.getsize(output_filename) / (1024 * 1024)  # Convertir a MB
    print(f"\n✓ Archivo verificado: {file_size:.2f} MB")
print("\n" + "=" * 60)
print("ANÁLISIS COMPLETADO EXITOSAMENTE")
print("=" * 60)


GUARDANDO RESULTADOS EN EXCEL

✓ Archivo 'resultadosReto.xlsx' creado exitosamente

 Resumen de hojas creadas:
   1. conDiferencias - 86,512 filas
   2. conCantidad    - 86,512 filas
   3. antes20        - 70 filas
   4. pfizer         - 64,193 filas

✓ Archivo verificado: 17.25 MB

ANÁLISIS COMPLETADO EXITOSAMENTE


In [30]:
## 📥 Descargar el archivo resultadosReto.xlsx

# ============================================
# CELDA CÓDIGO - DESCARGA
# ============================================
# Descomentar para descargar el archivo en Google Colab

from google.colab import files
files.download('resultadosReto.xlsx')
print("✓ Descarga iniciada")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Descarga iniciada
